# CEVAE実験 v4：1次元 Z・val_ELBO によるハイパラ選択

- **DGP**: v2（Z が1次元, tanh 非線形リンク, d=8 proxy, tau に非線形項あり）
- **ハイパラ選択**: X-learner は CV、CEVAE は val_ELBO（どちらも真の τ を使わない）
- 上から順に実行するだけで結果が出る（約15〜20分）

In [ ]:
import warnings
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy.stats import uniform, randint
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from xgboost import XGBRegressor
import torch
import torch.nn as nn

warnings.filterwarnings("ignore", message=".*encountered in matmul.*", category=RuntimeWarning)


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def generate_toy_v2(N=20000, d=8, gamma=1.5, theta=2.0, alpha=1.0, beta=1.0, delta_nl=0.8,
                    sigma_x=3.5, sigma_y=1.0, link_scale=1.5, seed=0):
    """v2 DGP: Z ~ N(0,1), X = tanh(1.5*Z)*a + noise, tau = alpha + beta*Z + delta_nl*(Z^2-1)"""
    rng = np.random.default_rng(seed)
    Z = rng.normal(0, 1, size=N)
    g_z = np.tanh(link_scale * Z)
    a = rng.normal(1.0, 1.0, size=d)
    X = g_z[:, None] * a[None, :] + rng.normal(0, sigma_x, size=(N, d))
    T = rng.binomial(1, sigmoid(gamma * Z))
    tau = alpha + beta * Z + delta_nl * (Z ** 2 - 1)
    base = theta * Z
    Y0 = base + rng.normal(0, sigma_y, size=N)
    Y1 = base + tau + rng.normal(0, sigma_y, size=N)
    Y = np.where(T == 1, Y1, Y0)
    return X, T, Y, Y0, Y1, tau, Z

In [ ]:
DATA_SEED = 0
X, T, Y, Y0, Y1, tau, Z = generate_toy_v2(seed=DATA_SEED)
d = X.shape[1]
N = len(Y)

proxy_cols = [f"X{j+1}" for j in range(d)]
df_full = pd.DataFrame(X, columns=proxy_cols)
df_full["T"] = T; df_full["Y"] = Y; df_full["tau"] = tau

df_train, df_test = train_test_split(df_full, test_size=0.3,
                                     random_state=DATA_SEED, stratify=df_full["T"])

true_ate = tau.mean()
naive_ate = Y[T==1].mean() - Y[T==0].mean()
corr_t_z = np.corrcoef(T, Z)[0, 1]
Xd = np.column_stack([np.ones(N), X])
coef, _, _, _ = np.linalg.lstsq(Xd, Z, rcond=None)
r2_zx = 1 - np.sum((Z - Xd @ coef)**2) / np.sum((Z - Z.mean())**2)

print(f"N={N}, d={d}")
print(f"真のATE           : {true_ate:.3f}")
print(f"ナイーブATE       : {naive_ate:.3f}  (バイアス={naive_ate - true_ate:+.3f})")
print(f"corr(T, Z)        : {corr_t_z:.3f}")
print(f"R^2(Z|X) 線形回帰 : {r2_zx:.3f}")

## X-learner ハイパラ探索（data_seed=0）

In [ ]:
XGB_PARAM_DIST = {
    "n_estimators": randint(300, 2500),
    "learning_rate": uniform(0.01, 0.19),
    "max_depth": randint(2, 9),
    "min_child_weight": randint(1, 10),
    "subsample": uniform(0.7, 0.3),
    "colsample_bytree": uniform(0.7, 0.3),
    "reg_lambda": uniform(0.0, 10.0),
}


def search_xgb(X_in, y_in, n_iter=50, cv=5, seed=0):
    s = RandomizedSearchCV(
        XGBRegressor(objective="reg:squarederror", random_state=seed, verbosity=0),
        param_distributions=XGB_PARAM_DIST,
        n_iter=n_iter, cv=cv, scoring="neg_mean_squared_error",
        n_jobs=-1, random_state=seed,
    )
    s.fit(X_in, y_in)
    return s.best_estimator_


def run_xlearner(df_tr, df_te, pcols, seed=0):
    df0 = df_tr[df_tr["T"] == 0]; df1 = df_tr[df_tr["T"] == 1]
    m0 = search_xgb(df0[pcols], df0["Y"], seed=seed)
    m1 = search_xgb(df1[pcols], df1["Y"], seed=seed)
    D0 = m1.predict(df0[pcols]) - df0["Y"].values
    D1 = df1["Y"].values - m0.predict(df1[pcols])
    x0 = search_xgb(df0[pcols], D0, seed=seed)
    x1 = search_xgb(df1[pcols], D1, seed=seed)
    prop = LogisticRegression(max_iter=1000).fit(df_tr[pcols], df_tr["T"])
    g0 = prop.predict_proba(df_te[pcols])[:, 0]
    tau_pred = g0 * x0.predict(df_te[pcols]) + (1 - g0) * x1.predict(df_te[pcols])
    return tau_pred, (m0, m1, x0, x1, prop)


print("X-learner ハイパラ探索中（data_seed=0, n_iter=50, cv=5）...")
xl_tau_pred, xl_models_fixed = run_xlearner(df_train, df_test, proxy_cols, seed=0)
xl_pehe_s0 = np.sqrt(np.mean((df_test["tau"].values - xl_tau_pred) ** 2))
print(f"X-learner √PEHE (data_seed=0): {xl_pehe_s0:.3f}")

## ネットワーク・関数定義

In [ ]:
class DecoderX(nn.Module):
    def __init__(self, z_dim, d, hidden):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(z_dim, hidden), nn.ReLU(), nn.Linear(hidden, d))
    def forward(self, z): return self.net(z)

class DecoderT(nn.Module):
    def __init__(self, z_dim, hidden):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(z_dim, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, z): return self.net(z).squeeze(-1)

class DecoderY(nn.Module):
    def __init__(self, z_dim, hidden):
        super().__init__()
        self.head0 = nn.Sequential(nn.Linear(z_dim, hidden), nn.ReLU(), nn.Linear(hidden, 1))
        self.head1 = nn.Sequential(nn.Linear(z_dim, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, z): return self.head0(z).squeeze(-1), self.head1(z).squeeze(-1)

class EncoderT(nn.Module):
    def __init__(self, d, hidden):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, x): return self.net(x).squeeze(-1)

class EncoderY(nn.Module):
    def __init__(self, d, hidden):
        super().__init__()
        self.head0 = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(), nn.Linear(hidden, 1))
        self.head1 = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, x): return self.head0(x).squeeze(-1), self.head1(x).squeeze(-1)

class EncoderZ(nn.Module):
    def __init__(self, d, z_dim, hidden):
        super().__init__()
        self.trunk   = nn.Sequential(nn.Linear(d, hidden), nn.ReLU())
        self.head0   = nn.Sequential(nn.Linear(hidden + 1, hidden), nn.ReLU())
        self.head1   = nn.Sequential(nn.Linear(hidden + 1, hidden), nn.ReLU())
        self.mu0     = nn.Linear(hidden, z_dim)
        self.logvar0 = nn.Linear(hidden, z_dim)
        self.mu1     = nn.Linear(hidden, z_dim)
        self.logvar1 = nn.Linear(hidden, z_dim)
    def forward(self, x, t, y):
        h  = self.trunk(x)
        hy = torch.cat([h, y.unsqueeze(-1)], dim=-1)
        h0, h1 = self.head0(hy), self.head1(hy)
        mu0, lv0 = self.mu0(h0), self.logvar0(h0)
        mu1, lv1 = self.mu1(h1), self.logvar1(h1)
        tc = t.unsqueeze(-1)
        return torch.where(tc == 1, mu1, mu0), torch.where(tc == 1, lv1, lv0)


def reparameterize(mu, logvar):
    return mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)

def kl_to_standard_normal(mu, logvar):
    return (-0.5 * (1 + logvar - mu.pow(2) - logvar.exp())).sum(dim=-1).mean()

def linear_kl_anneal(step, total_steps):
    return 1.0 if total_steps <= 0 else min(1.0, step / total_steps)


@dataclass
class CEVAEConfig:
    z_dim: int = 1
    hidden: int = 64
    beta_kl_max: float = 1.0
    kl_anneal_epochs: int = 75
    epochs: int = 150
    batch_size: int = 256
    lr: float = 1e-3
    weight_decay: float = 0.0
    seed: int = 0


def build_cevae(cfg, d):
    torch.manual_seed(cfg.seed)
    return {
        "decoder_x": DecoderX(cfg.z_dim, d, cfg.hidden),
        "decoder_t": DecoderT(cfg.z_dim, cfg.hidden),
        "decoder_y": DecoderY(cfg.z_dim, cfg.hidden),
        "encoder_t": EncoderT(d, cfg.hidden),
        "encoder_y": EncoderY(d, cfg.hidden),
        "encoder_z": EncoderZ(d, cfg.z_dim, cfg.hidden),
    }


def cevae_loss_fn(nets, x, t, y, beta_kl):
    mu_z, logvar_z = nets["encoder_z"](x, t, y)
    z = reparameterize(mu_z, logvar_z)
    x_hat = nets["decoder_x"](z)
    t_logit_p = nets["decoder_t"](z)
    y0p, y1p = nets["decoder_y"](z)
    y_hat_p = torch.where(t == 1, y1p, y0p)
    recon_x = nn.functional.mse_loss(x_hat, x)
    recon_t = nn.functional.binary_cross_entropy_with_logits(t_logit_p, t)
    recon_y = nn.functional.mse_loss(y_hat_p, y)
    kl = kl_to_standard_normal(mu_z, logvar_z)
    t_logit_q = nets["encoder_t"](x)
    y0q, y1q = nets["encoder_y"](x)
    y_hat_q = torch.where(t == 1, y1q, y0q)
    aux_t = nn.functional.binary_cross_entropy_with_logits(t_logit_q, t)
    aux_y = nn.functional.mse_loss(y_hat_q, y)
    return recon_x + recon_t + recon_y + beta_kl * kl + aux_t + aux_y


def train_cevae(nets, cfg, x_tr, t_tr, y_tr):
    torch.manual_seed(cfg.seed)
    all_params = [p for net in nets.values() for p in net.parameters()]
    opt = torch.optim.Adam(all_params, lr=cfg.lr, weight_decay=cfg.weight_decay)
    n = x_tr.shape[0]
    n_batches = max(1, n // cfg.batch_size)
    for epoch in range(cfg.epochs):
        beta_kl = cfg.beta_kl_max * linear_kl_anneal(epoch, cfg.kl_anneal_epochs)
        perm = torch.randperm(n)
        for b in range(n_batches):
            idx = perm[b * cfg.batch_size:(b + 1) * cfg.batch_size]
            opt.zero_grad()
            cevae_loss_fn(nets, x_tr[idx], t_tr[idx], y_tr[idx], beta_kl).backward()
            opt.step()


def compute_val_elbo(nets, x_val, t_val, y_val):
    with torch.no_grad():
        return cevae_loss_fn(nets, x_val, t_val, y_val, beta_kl=1.0).item()


def estimate_ite(nets, x):
    with torch.no_grad():
        t_hat = (torch.sigmoid(nets["encoder_t"](x)) > 0.5).float()
        y0q, y1q = nets["encoder_y"](x)
        y_hat = torch.where(t_hat == 1, y1q, y0q)
        mu_z, _ = nets["encoder_z"](x, t_hat, y_hat)
        y0p, y1p = nets["decoder_y"](mu_z)
        return (y1p - y0p).numpy()


def z_corr_general(nets, x, t, y, Z_true):
    """z_dim が何次元でも使える z 回収スコア（真の Z の各次元に最も相関する mu_z 次元を選んで平均）"""
    with torch.no_grad():
        mu_z, _ = nets["encoder_z"](x, t, y)
    mz = mu_z.numpy()
    if Z_true.ndim == 1:
        Z_true = Z_true[:, None]
    d_true, z_dim = Z_true.shape[1], mz.shape[1]
    scores = [max(abs(np.corrcoef(mz[:, k], Z_true[:, j])[0, 1]) for k in range(z_dim))
              for j in range(d_true)]
    return float(np.mean(scores))

## CEVAE ハイパラ探索（data_seed=0, val_ELBO 基準）

訓練データを inner_train (80%) / val (20%) に分けてランダムサーチ。
val_ELBO が最小の設定を全 seed に流用する（真の τ を一切使わない）。

In [ ]:

df_inner, df_val = train_test_split(df_train, test_size=0.2,
                                    random_state=0, stratify=df_train["T"])

def df_to_tensors(df):
    return (
        torch.tensor(df[proxy_cols].values, dtype=torch.float32),
        torch.tensor(df["T"].values,        dtype=torch.float32),
        torch.tensor(df["Y"].values,        dtype=torch.float32),
    )

x_inner, t_inner, y_inner = df_to_tensors(df_inner)
x_val,   t_val,   y_val   = df_to_tensors(df_val)

# 真の Z が1次元なので z_dim=1 に固定
SEARCH_SPACE = {
    "hidden":           [32, 64, 128],
    "beta_kl_max":      [0.1, 0.3, 0.5, 1.0],
    "kl_anneal_epochs": [50, 100, 150],
    "epochs":           [150, 300],
    "weight_decay":     [0.0, 1e-4, 1e-3],
    "lr":               [1e-3, 5e-4, 3e-4],
}
# 総組み合わせ数 3×4×3×2×3×3 = 648、N_ITER=150 で約23%をカバー
N_ITER = 150

rng_search = random.Random(0)
search_records = []

print(f"CEVAE ランダムサーチ（z_dim=1固定, n_iter={N_ITER}）...")
for i in range(N_ITER):
    cfg = CEVAEConfig(
        z_dim=1,
        hidden=rng_search.choice(SEARCH_SPACE["hidden"]),
        beta_kl_max=rng_search.choice(SEARCH_SPACE["beta_kl_max"]),
        kl_anneal_epochs=rng_search.choice(SEARCH_SPACE["kl_anneal_epochs"]),
        epochs=rng_search.choice(SEARCH_SPACE["epochs"]),
        weight_decay=rng_search.choice(SEARCH_SPACE["weight_decay"]),
        lr=rng_search.choice(SEARCH_SPACE["lr"]),
        seed=0,
    )
    nets = build_cevae(cfg, d)
    train_cevae(nets, cfg, x_inner, t_inner, y_inner)
    val_elbo = compute_val_elbo(nets, x_val, t_val, y_val)
    search_records.append({"iter": i + 1, "val_elbo": val_elbo,
                           "z_dim": cfg.z_dim, "hidden": cfg.hidden,
                           "beta_kl_max": cfg.beta_kl_max,
                           "kl_anneal_epochs": cfg.kl_anneal_epochs,
                           "epochs": cfg.epochs, "weight_decay": cfg.weight_decay,
                           "lr": cfg.lr})
    print(f"  [{i+1:3d}/{N_ITER}] val_elbo={val_elbo:.3f}  "
          f"h={cfg.hidden} beta={cfg.beta_kl_max} "
          f"anneal={cfg.kl_anneal_epochs} ep={cfg.epochs} wd={cfg.weight_decay} lr={cfg.lr:.0e}")

df_search = pd.DataFrame(search_records).sort_values("val_elbo")
best_row = df_search.iloc[0]
print(f"\n=== ベスト設定（上位5件）===")
print(df_search.head(5).to_string(index=False))

BEST_CEVAE_CFG = CEVAEConfig(
    z_dim=1,
    hidden=int(best_row["hidden"]),
    beta_kl_max=float(best_row["beta_kl_max"]),
    kl_anneal_epochs=int(best_row["kl_anneal_epochs"]),
    epochs=int(best_row["epochs"]),
    weight_decay=float(best_row["weight_decay"]),
    lr=float(best_row["lr"]),
    seed=0,
)
print(f"\nBEST_CEVAE_CFG: {BEST_CEVAE_CFG}")


## マルチシード実験（data_seed 0〜9）

両手法とも data_seed=0 で探索したパラメータを全 seed に流用。

In [ ]:

TOP_K = 10
N_DATA_SEEDS = 10
records = []

# 上位K件の設定を取得
top_cfgs = []
for _, row in df_search.head(TOP_K).iterrows():
    top_cfgs.append(CEVAEConfig(
        z_dim=1,
        hidden=int(row["hidden"]),
        beta_kl_max=float(row["beta_kl_max"]),
        kl_anneal_epochs=int(row["kl_anneal_epochs"]),
        epochs=int(row["epochs"]),
        weight_decay=float(row["weight_decay"]),
        lr=float(row["lr"]),
        seed=0,
    ))

print(f"上位 {TOP_K} 件:")
for i, c in enumerate(top_cfgs):
    print(f"  [{i+1:2d}] h={c.hidden} beta={c.beta_kl_max} "
          f"anneal={c.kl_anneal_epochs} ep={c.epochs} wd={c.weight_decay} lr={c.lr:.0e}")

for ds in range(N_DATA_SEEDS):
    print(f"\ndata_seed={ds}", flush=True)
    Xs, Ts, Ys, _, _, taus, Zs = generate_toy_v2(seed=ds)

    df_s = pd.DataFrame(Xs, columns=proxy_cols)
    df_s["T"] = Ts; df_s["Y"] = Ys; df_s["tau"] = taus
    df_tr_s, df_te_s = train_test_split(df_s, test_size=0.3,
                                        random_state=ds, stratify=df_s["T"])
    tau_true_s = df_te_s["tau"].values
    te_idx = df_te_s.index.tolist()
    tr_idx = df_tr_s.index.tolist()
    Z_te_s = Zs[te_idx]

    # このシード用の inner/val 分割（ハイパラ選択に使う）
    df_inner_s, df_val_s = train_test_split(df_tr_s, test_size=0.2,
                                            random_state=ds, stratify=df_tr_s["T"])
    x_inner_s, t_inner_s, y_inner_s = df_to_tensors(df_inner_s)
    x_val_s,   t_val_s,   y_val_s   = df_to_tensors(df_val_s)

    # ── X-learner（固定パラメータで再学習）
    m0_f, m1_f, x0_f, x1_f, _ = xl_models_fixed
    df0s = df_tr_s[df_tr_s["T"]==0]; df1s = df_tr_s[df_tr_s["T"]==1]
    m0s = m0_f.__class__(**m0_f.get_params()).fit(df0s[proxy_cols], df0s["Y"])
    m1s = m1_f.__class__(**m1_f.get_params()).fit(df1s[proxy_cols], df1s["Y"])
    D0s = m1s.predict(df0s[proxy_cols]) - df0s["Y"].values
    D1s = df1s["Y"].values - m0s.predict(df1s[proxy_cols])
    x0s = x0_f.__class__(**x0_f.get_params()).fit(df0s[proxy_cols], D0s)
    x1s = x1_f.__class__(**x1_f.get_params()).fit(df1s[proxy_cols], D1s)
    props = LogisticRegression(max_iter=1000).fit(df_tr_s[proxy_cols], df_tr_s["T"])
    g0s = props.predict_proba(df_te_s[proxy_cols])[:, 0]
    tau_xl = g0s * x0s.predict(df_te_s[proxy_cols]) + (1-g0s) * x1s.predict(df_te_s[proxy_cols])
    xl_pehe_s = np.sqrt(np.mean((tau_true_s - tau_xl)**2))

    # ── CEVAE：上位10件を inner で試して val_ELBO が最小のものを選択
    best_elbo = float("inf")
    best_cfg_idx = 0
    for k, base_cfg in enumerate(top_cfgs):
        cfg_tmp = CEVAEConfig(
            z_dim=1, hidden=base_cfg.hidden,
            beta_kl_max=base_cfg.beta_kl_max, kl_anneal_epochs=base_cfg.kl_anneal_epochs,
            epochs=base_cfg.epochs, weight_decay=base_cfg.weight_decay,
            lr=base_cfg.lr, seed=ds,
        )
        nets_tmp = build_cevae(cfg_tmp, d)
        train_cevae(nets_tmp, cfg_tmp, x_inner_s, t_inner_s, y_inner_s)
        elbo = compute_val_elbo(nets_tmp, x_val_s, t_val_s, y_val_s)
        print(f"  cfg[{k+1:2d}] val_elbo={elbo:.3f}", end="")
        if elbo < best_elbo:
            best_elbo = elbo
            best_cfg_idx = k
    print(f"  → 採用: cfg[{best_cfg_idx+1}]")

    # ── 最良設定で全訓練データを使って再学習 → 評価
    best_base = top_cfgs[best_cfg_idx]
    cfg_final = CEVAEConfig(
        z_dim=1, hidden=best_base.hidden,
        beta_kl_max=best_base.beta_kl_max, kl_anneal_epochs=best_base.kl_anneal_epochs,
        epochs=best_base.epochs, weight_decay=best_base.weight_decay,
        lr=best_base.lr, seed=ds,
    )
    xt = torch.tensor(Xs, dtype=torch.float32)
    tt = torch.tensor(Ts, dtype=torch.float32)
    yt = torch.tensor(Ys, dtype=torch.float32)
    nets_s = build_cevae(cfg_final, d)
    train_cevae(nets_s, cfg_final, xt[tr_idx], tt[tr_idx], yt[tr_idx])
    tau_cv = estimate_ite(nets_s, xt[te_idx])
    cv_pehe_s = np.sqrt(np.mean((tau_true_s - tau_cv)**2))
    zc_s = z_corr_general(nets_s, xt[te_idx], tt[te_idx], yt[te_idx], Z_te_s)

    records.append({"data_seed": ds, "xl_pehe": xl_pehe_s,
                    "cv_pehe": cv_pehe_s, "z_corr": zc_s, "best_cfg": best_cfg_idx + 1})
    print(f"  → XL={xl_pehe_s:.3f}  CEVAE={cv_pehe_s:.3f}  z_corr={zc_s:.3f}")

df_res = pd.DataFrame(records)
print("\n=== 集計 ===")
print(df_res[["xl_pehe", "cv_pehe", "z_corr"]].agg(["mean", "std"]).round(3))
print(f"\nCEVAE 勝利(seed別): {(df_res.cv_pehe < df_res.xl_pehe).sum()}/10 seed")


In [ ]:

# ── 結果表示（パワポ用）────────────────────────────────────────────

# seed 別結果
df_display = df_res[["data_seed", "xl_pehe", "cv_pehe", "z_corr"]].copy()
df_display.columns = ["seed", "X-learner √PEHE", "CEVAE √PEHE", "z_corr"]
df_display = df_display.round(3)
display(df_display)

# 集計行を追加
summary = pd.DataFrame([{
    "seed": "mean",
    "X-learner √PEHE": round(df_res["xl_pehe"].mean(), 3),
    "CEVAE √PEHE":     round(df_res["cv_pehe"].mean(), 3),
    "z_corr":          round(df_res["z_corr"].mean(), 3),
}, {
    "seed": "std",
    "X-learner √PEHE": round(df_res["xl_pehe"].std(), 3),
    "CEVAE √PEHE":     round(df_res["cv_pehe"].std(), 3),
    "z_corr":          round(df_res["z_corr"].std(), 3),
}])
display(summary)

wins = (df_res["cv_pehe"] < df_res["xl_pehe"]).sum()
print(f"CEVAE 勝利: {wins}/10 seed")


In [ ]:
import subprocess
subprocess.run(["osascript", "-e", 'tell application "System Events" to sleep'])


## 診断：失敗した data_seed (3, 6, 9) は「初期値運」か「データが難しい」か

上のマルチシード実験で z_corr が大きく落ちた3つの data_seed について、
**採用されたハイパラは固定したまま、model_seed だけを 0〜9 で独立に振り直す**。

- ある model_seed で z_corr が 0.85 を超える → **初期値運が原因**（初期値ガチャで救える）
- 何回振っても z_corr が低いまま → **そのデータ自体が識別困難**（初期値の問題ではない）

真の τ を使わない設計方針を維持するため、model_seed 選択の判断材料としても val_ELBO を併記する
（実運用でどれだけ「カンニングなしで」救えるかも見るため）。

In [ ]:
FAILED_SEEDS = [3, 6, 9]
N_MODEL_SEEDS = 10

diag_records = []

for ds in FAILED_SEEDS:
    print(f"\n=== data_seed={ds} ===", flush=True)
    Xs, Ts, Ys, _, _, taus, Zs = generate_toy_v2(seed=ds)

    df_s = pd.DataFrame(Xs, columns=proxy_cols)
    df_s["T"] = Ts; df_s["Y"] = Ys; df_s["tau"] = taus
    df_tr_s, df_te_s = train_test_split(df_s, test_size=0.3,
                                        random_state=ds, stratify=df_s["T"])
    tau_true_s = df_te_s["tau"].values
    te_idx = df_te_s.index.tolist()
    tr_idx = df_tr_s.index.tolist()
    Z_te_s = Zs[te_idx]

    # val_ELBO用のinner/val分割（cell[10]と同じ切り方）
    df_inner_s, df_val_s = train_test_split(df_tr_s, test_size=0.2,
                                            random_state=ds, stratify=df_tr_s["T"])
    x_inner_s, t_inner_s, y_inner_s = df_to_tensors(df_inner_s)
    x_val_s,   t_val_s,   y_val_s   = df_to_tensors(df_val_s)

    xt = torch.tensor(Xs, dtype=torch.float32)
    tt = torch.tensor(Ts, dtype=torch.float32)
    yt = torch.tensor(Ys, dtype=torch.float32)

    # cell[10]でこのdata_seedが実際に採用したハイパラを固定して使う（ハイパラは触らない）
    best_cfg_idx_for_ds = int(df_res.loc[df_res["data_seed"] == ds, "best_cfg"].iloc[0]) - 1
    base_cfg = top_cfgs[best_cfg_idx_for_ds]
    print(f"  採用ハイパラ(cfg[{best_cfg_idx_for_ds+1}]): h={base_cfg.hidden} beta={base_cfg.beta_kl_max} "
          f"anneal={base_cfg.kl_anneal_epochs} ep={base_cfg.epochs} "
          f"wd={base_cfg.weight_decay} lr={base_cfg.lr:.0e}")

    for ms in range(N_MODEL_SEEDS):
        cfg_ms = CEVAEConfig(
            z_dim=1, hidden=base_cfg.hidden,
            beta_kl_max=base_cfg.beta_kl_max, kl_anneal_epochs=base_cfg.kl_anneal_epochs,
            epochs=base_cfg.epochs, weight_decay=base_cfg.weight_decay,
            lr=base_cfg.lr, seed=ms,   # ← data_seedではなくmodel_seedだけを独立に振る
        )
        nets_ms = build_cevae(cfg_ms, d)
        train_cevae(nets_ms, cfg_ms, xt[tr_idx], tt[tr_idx], yt[tr_idx])

        val_elbo_ms = compute_val_elbo(nets_ms, x_val_s, t_val_s, y_val_s)
        tau_pred_ms = estimate_ite(nets_ms, xt[te_idx])
        pehe_ms = np.sqrt(np.mean((tau_true_s - tau_pred_ms) ** 2))
        zc_ms = z_corr_general(nets_ms, xt[te_idx], tt[te_idx], yt[te_idx], Z_te_s)

        diag_records.append({"data_seed": ds, "model_seed": ms,
                             "val_elbo": val_elbo_ms, "pehe": pehe_ms, "z_corr": zc_ms})
        status = "✅" if zc_ms > 0.85 else "❌"
        print(f"  model_seed={ms}  val_elbo={val_elbo_ms:.3f}  "
              f"PEHE={pehe_ms:.3f}  z_corr={zc_ms:.3f}  {status}")

df_diag = pd.DataFrame(diag_records)

print("\n=== data_seedごとの結果まとめ ===")
for ds in FAILED_SEEDS:
    sub = df_diag[df_diag["data_seed"] == ds]
    n_rescued = (sub["z_corr"] > 0.85).sum()
    best_by_elbo = sub.loc[sub["val_elbo"].idxmin()]
    print(f"data_seed={ds}: z_corr>0.85のmodel_seed数 = {n_rescued}/{N_MODEL_SEEDS}  "
          f"| val_ELBO最小で選んだ場合 → model_seed={int(best_by_elbo['model_seed'])}, "
          f"z_corr={best_by_elbo['z_corr']:.3f}, PEHE={best_by_elbo['pehe']:.3f}")

display(df_diag.round(3))

## val_ELBOに代わる、真のτを使わない選択信号の検証

val_ELBOは「成功(z_corr≈0.93, √PEHE≈1.3〜1.7)」と「崩壊(z_corr≈0.47, √PEHE≈6〜8)」の二山を
区別できないことが分かった（data_seed=9では崩壊runの方がval_ELBOが最良だった＝非識別性の実測）。

ここでは、val_ELBOの代わりに使える「真のτ・真のZを一切使わない」選択信号の候補を探す：

- **mse_diff = mse_p − mse_q**（`CEVAE診断_seed3.ipynb`由来）
  - `mse_q`：q(y|x,t) の予測誤差（zを介さない）
  - `mse_p`：test-timeフロー p(y|z,t) の予測誤差（zを推定してから予測。真のyは使わない）
  - 仮説：崩壊run（zが無情報）では `mse_p` が悪化し、`mse_diff` が正に大きくなるはず
- **z情報量指標**（posterior collapse検出用、観測された t, y だけを使う）：
  `z_var`（mu_zの分散）、`z_t_corr`=|corr(mu_z, T)|、`z_y_corr`=|corr(mu_z, Y)|

真のτ・真のZは「答え合わせ」列（`sqrt_pehe`, `z_corr`）にのみ使い、選択信号には一切使わない。
data_seed 3/6/9 × model_seed 0〜9 の30 runで、ハイパラは各data_seedの採用済み設定を固定する。

In [ ]:
def compute_y_mse_diag(nets, x_te, t_te_obs, y_te_np):
    """真のτ・真のZを一切使わない、テスト時に計算可能な診断指標。
    mse_q: q(y|x,t)（zを介さない）の予測誤差。mse_p: test-timeフロー p(y|z,t)（zを推定してから予測）の予測誤差。
    mse_diff = mse_p - mse_q。仮説: 崩壊run(zが無情報)ではmse_pが悪化しmse_diffが正に大きくなる。
    """
    with torch.no_grad():
        y0_q, y1_q = nets["encoder_y"](x_te)
        y_q_hat = torch.where(t_te_obs == 1, y1_q, y0_q)

        t_hat = (torch.sigmoid(nets["encoder_t"](x_te)) > 0.5).float()
        y0_for_z, y1_for_z = nets["encoder_y"](x_te)
        y_hat_for_z = torch.where(t_hat == 1, y1_for_z, y0_for_z)
        z_mu_te, _ = nets["encoder_z"](x_te, t_hat, y_hat_for_z)
        y0_p, y1_p = nets["decoder_y"](z_mu_te)
        y_p_hat = torch.where(t_te_obs == 1, y1_p, y0_p)

    y_q = y_q_hat.numpy()
    y_p = y_p_hat.numpy()
    mse_q = float(np.mean((y_te_np - y_q) ** 2))
    mse_p = float(np.mean((y_te_np - y_p) ** 2))
    return mse_q, mse_p, mse_p - mse_q


def z_info_signals(nets, x_obs, t_obs, y_obs):
    """真のτ・真のZを使わない、zの情報量診断（posterior collapse検出用）。
    観測された t, y（学習時に実際に使える値）だけを使う。
    """
    with torch.no_grad():
        mu_z, _ = nets["encoder_z"](x_obs, t_obs, y_obs)
    mz = mu_z.numpy().reshape(-1)
    z_var = float(np.var(mz))
    z_t_corr = float(abs(np.corrcoef(mz, t_obs.numpy())[0, 1]))
    z_y_corr = float(abs(np.corrcoef(mz, y_obs.numpy())[0, 1]))
    return z_var, z_t_corr, z_y_corr


def confusion_at_threshold(df, signal_col, threshold, direction="above", true_col="collapsed"):
    """direction='above'ならsignal>=thresholdを崩壊予測、'below'ならsignal<=thresholdを崩壊予測とする。"""
    true_collapsed = df[true_col]
    pred_collapsed = df[signal_col] >= threshold if direction == "above" else df[signal_col] <= threshold
    tp = int((pred_collapsed & true_collapsed).sum())
    fn = int((~pred_collapsed & true_collapsed).sum())
    fp = int((pred_collapsed & ~true_collapsed).sum())
    tn = int((~pred_collapsed & ~true_collapsed).sum())
    return {"signal": signal_col, "direction": direction, "threshold": threshold,
            "崩壊を正しく検出(TP)": tp, "崩壊を見逃す(FN)": fn,
            "成功を誤って除外(FP)": fp, "成功を正しく残す(TN)": tn, "errors": fn + fp}


def best_threshold(df, signal_col, true_col="collapsed"):
    """真のラベルに対して最も分離が良い閾値・方向を全探索で選ぶ（分離可能性そのものを評価する目的）。"""
    values = np.sort(df[signal_col].unique())
    candidates = (values[:-1] + values[1:]) / 2 if len(values) > 1 else values
    best = None
    for direction in ["above", "below"]:
        for th in candidates:
            res = confusion_at_threshold(df, signal_col, th, direction, true_col)
            if best is None or res["errors"] < best["errors"]:
                best = res
    return best

In [ ]:
FAILED_SEEDS = [3, 6, 9]
N_MODEL_SEEDS = 10

signal_records = []

for ds in FAILED_SEEDS:
    print(f"\n=== data_seed={ds} ===", flush=True)
    Xs, Ts, Ys, _, _, taus, Zs = generate_toy_v2(seed=ds)

    df_s = pd.DataFrame(Xs, columns=proxy_cols)
    df_s["T"] = Ts; df_s["Y"] = Ys; df_s["tau"] = taus
    df_tr_s, df_te_s = train_test_split(df_s, test_size=0.3,
                                        random_state=ds, stratify=df_s["T"])
    tau_true_s = df_te_s["tau"].values          # ← 答え合わせ専用
    te_idx = df_te_s.index.tolist()
    tr_idx = df_tr_s.index.tolist()
    Z_te_s = Zs[te_idx]                          # ← 答え合わせ専用

    # val_ELBO（ベースライン選択信号）用のinner/val分割（ds単位、cell[10]と同じ切り方）
    df_inner_s, df_val_s = train_test_split(df_tr_s, test_size=0.2,
                                            random_state=ds, stratify=df_tr_s["T"])
    x_val_s, t_val_s, y_val_s = df_to_tensors(df_val_s)

    xt = torch.tensor(Xs, dtype=torch.float32)
    tt = torch.tensor(Ts, dtype=torch.float32)
    yt = torch.tensor(Ys, dtype=torch.float32)
    x_te_s, t_te_s = xt[te_idx], tt[te_idx]
    y_te_s_np = df_te_s["Y"].values

    # cell[10]でこのdata_seedが実際に採用したハイパラを固定して使う
    best_cfg_idx_for_ds = int(df_res.loc[df_res["data_seed"] == ds, "best_cfg"].iloc[0]) - 1
    base_cfg = top_cfgs[best_cfg_idx_for_ds]

    for ms in range(N_MODEL_SEEDS):
        cfg_ms = CEVAEConfig(
            z_dim=1, hidden=base_cfg.hidden,
            beta_kl_max=base_cfg.beta_kl_max, kl_anneal_epochs=base_cfg.kl_anneal_epochs,
            epochs=base_cfg.epochs, weight_decay=base_cfg.weight_decay,
            lr=base_cfg.lr, seed=ms,   # ← data_seedとは独立にmodel_seedだけ振る
        )
        nets_ms = build_cevae(cfg_ms, d)
        train_cevae(nets_ms, cfg_ms, xt[tr_idx], tt[tr_idx], yt[tr_idx])

        # ── 答え合わせ専用（真のτ・真のZを使う。選択信号には絶対使わない）──
        tau_pred_ms = estimate_ite(nets_ms, x_te_s)
        pehe_ms = float(np.sqrt(np.mean((tau_true_s - tau_pred_ms) ** 2)))
        zc_ms = z_corr_general(nets_ms, x_te_s, t_te_s, yt[te_idx], Z_te_s)

        # ── ベースライン選択信号：val_ELBO ──
        val_elbo_ms = compute_val_elbo(nets_ms, x_val_s, t_val_s, y_val_s)

        # ── 候補信号1：mse_diff（真のτ・Zを使わない） ──
        mse_q, mse_p, mse_diff = compute_y_mse_diag(nets_ms, x_te_s, t_te_s, y_te_s_np)

        # ── 候補信号2：z情報量（真のτ・Zを使わない。観測されたtrainのt, yを使う） ──
        z_var, z_t_corr, z_y_corr = z_info_signals(nets_ms, xt[tr_idx], tt[tr_idx], yt[tr_idx])

        signal_records.append({
            "data_seed": ds, "model_seed": ms,
            "val_elbo": val_elbo_ms,
            "sqrt_pehe": pehe_ms, "z_corr": zc_ms,          # 答え合わせ専用
            "mse_q": mse_q, "mse_p": mse_p, "mse_diff": mse_diff,
            "z_var": z_var, "z_t_corr": z_t_corr, "z_y_corr": z_y_corr,
        })
        collapsed = zc_ms <= 0.85
        print(f"  ms={ms}  PEHE={pehe_ms:.3f}  z_corr={zc_ms:.3f}  elbo={val_elbo_ms:.3f}  "
              f"mse_diff={mse_diff:+.4f}  z_var={z_var:.4f}  z_t={z_t_corr:.3f}  z_y={z_y_corr:.3f}  "
              f"{'❌崩壊' if collapsed else '✅'}")

df_signal = pd.DataFrame(signal_records)
df_signal["collapsed"] = df_signal["z_corr"] <= 0.85

print(f"\n合計 {len(df_signal)} run中、崩壊 {int(df_signal['collapsed'].sum())} run")
print(df_signal.groupby("data_seed")["collapsed"].sum().astype(int).rename("崩壊run数(/10)"))
display(df_signal.round(4))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
signal_cols = ["mse_diff", "z_var", "z_t_corr", "z_y_corr"]
titles = ["mse_diff = mse_p - mse_q", "z_var (mu_zの分散)", "|corr(mu_z, T)|", "|corr(mu_z, Y)|"]

for ax, col, title in zip(axes, signal_cols, titles):
    for collapsed, color, label in [(False, "#2ecc71", "成功"), (True, "#e74c3c", "崩壊")]:
        sub = df_signal[df_signal["collapsed"] == collapsed]
        ax.scatter(sub[col], sub["sqrt_pehe"], c=color, label=label,
                   alpha=0.85, edgecolor="black", linewidth=0.4, s=60)
    ax.set_xlabel(col)
    ax.set_ylabel("√PEHE")
    ax.set_title(title)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
candidate_signals = ["val_elbo", "mse_diff", "z_var", "z_t_corr", "z_y_corr"]
comparison = pd.DataFrame([best_threshold(df_signal, c) for c in candidate_signals])
comparison = comparison.sort_values("errors").reset_index(drop=True)

print(f"(参考) 全30run中、真の崩壊run数 = {int(df_signal['collapsed'].sum())} / 30")
print("各信号の『最良の閾値』での混同行列（val_elboがベースライン。errorsが小さいほど良い分離）:\n")
display(comparison[["signal", "direction", "threshold",
                     "崩壊を正しく検出(TP)", "崩壊を見逃す(FN)",
                     "成功を誤って除外(FP)", "成功を正しく残す(TN)", "errors"]])